# **Imports**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as patches
import numpy as np
import os
import datetime
import math
import pickle
from dataclasses import dataclass

!pip install backtrader
%matplotlib inline
import backtrader as bt

# --- General Plotting Setup (run this once in your notebook/script) ---
plt.rcParams['figure.figsize'] = [18, 10] # Default figure size
plt.rcParams['figure.dpi'] = 100 # Default DPI


In [ ]:
print("backtrader:",bt.__version__)
print("numpy",np.__version__)
print("pandas",pd.__version__)

In [ ]:
# %load_ext autoreload
# %autoreload 2
# import importlib
# importlib.reload(run_results)


In [ ]:
%cd /content/
!rm -r crypto-bt-strategy
!git clone https://github.com/ssnsbr/crypto-bt-strategy.git
!ls
%cd crypto-bt-strategy
# !cp -r crypto-bt-strategy/* .
!ls


In [ ]:
from backtrader_extended.sizers.ScalperMartingaleSizer import ScalperMartingaleSizer
from backtrader_extended.strategies.FastScalperStrategy import FastScalperStrategy
from backtrader_extended.strategies.FiboMartingaleStrategy import FiboMartingaleStrategy
from utils.data_utils import *
from utils.plotting_utils import *
from riskmanagers.NoneRiskManagement import NoneRiskManagement
from backtrader_extended.strategies.Base import BaseTradingStrategy

from riskmanagers.ABCRiskManagement import AbstractRiskManagement

from run_results.runner import *
from run_results.runner_utils import *
import backtrader as bt
from utils.utils import format_marketcap, format_price_to_marketcap
from utils.utils import *
from run_results.analys_results import *
from run_results.custom_analyzers import BACounterAnalyzer, CashHistoryAnalyzer, TradeDurationAnalyzer
from backtrader_extended.sizers.FiboMartingaleSizer import FiboMartingaleSizer
from backtrader_extended.strategies import FiboMartingaleStrategy
from run_results.analyse_depth import *
# from utils.bounce_detector import BounceDetector
# from utils.liveliness_tracker import LivelinessTracker
from backtrader_extended.sizers.MartingaleSizer import MartingaleSizer


# **Data**

In [ ]:
# Connect to Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Define path to your CSV files
results_folder =  '/content/drive/MyDrive/charts/results/'

folder_path_15s = '/content/drive/MyDrive/charts/'+"15s/"
folder_path_1s = '/content/drive/MyDrive/charts/'+"1s/"

# List all CSV files
csv_files_1s = [folder_path_1s+f for f in os.listdir(folder_path_1s) if f.endswith('.csv')]
csv_files_15s = [folder_path_15s+f for f in os.listdir(folder_path_15s) if f.endswith('.csv')]

def new_files(list_csv):
    news = []
    for l in list_csv:
        timest = l.split("_")[-1].split(".")[0]
        if len(timest) ==13 and  int( timest ) > 1761409890000:
            news.append(l)
    return news

new_csv_files_1s =new_files(csv_files_1s)


print(f"Found {len(new_csv_files_1s)} new CSV files.")
print(new_csv_files_1s[0])

print(f"Found {len(csv_files_1s)} CSV files.")
print(csv_files_1s[0])

print(f"Found {len(csv_files_15s)} CSV files.")
print(csv_files_15s[0])

In [ ]:

index = 33
print(csv_files_1s[index],len(csv_files_1s[index]))
print(ready_df(pd.read_csv(csv_files_1s[index]),True).head(2))
df=pd.read_csv(csv_files_1s[index])
df = ready_df(df)
print(df.head())
print(df.tail())

df.head()
df[df["volume"]<0.01]


# Strategy Code

In [ ]:
from dataclasses import dataclass
from typing import List, Optional

@dataclass
class Pivot:
    index: int
    time: int
    price: float
    kind: str  # "high" or "low"


class ZigZag():
    """
    Percent-based ZigZag.
    reversal_pct = 5 means 5%
    """

    def __init__(self, reversal_pct: float):
        # super().__init__(key=f"zigzag_{reversal_pct}")
        self.reversal_pct = reversal_pct / 100.0

        self._candles_seen = 0
        self._direction: Optional[str] = None

        self._last_pivot_price: Optional[float] = None
        self._last_pivot_index: Optional[int] = None

        self._candidate_price: Optional[float] = None
        self._candidate_index: Optional[int] = None

        self._pivots: List[Pivot] = []

    # --------------------------------------------------------

    def update(self, candle ):
        price = candle.close
        idx = self._candles_seen
        self._candles_seen += 1

        if self._last_pivot_price is None:
            # First candle
            self._last_pivot_price = price
            self._last_pivot_index = idx
            self._candidate_price = price
            self._candidate_index = idx
            return

        # ----------------------------------------------------
        # No direction yet
        # ----------------------------------------------------
        if self._direction is None:
            change = (price - self._last_pivot_price) / self._last_pivot_price

            if abs(change) >= self.reversal_pct:
                if change > 0:
                    self._direction = "up"
                    kind = "low"
                else:
                    self._direction = "down"
                    kind = "high"

                # Confirm first pivot
                self._pivots.append(
                    Pivot(
                        index=self._last_pivot_index,
                        time=candle.time,
                        price=self._last_pivot_price,
                        kind=kind,
                    )
                )

                self._candidate_price = price
                self._candidate_index = idx

            return

        # ----------------------------------------------------
        # UP TREND
        # ----------------------------------------------------
        if self._direction == "up":
            if price > self._candidate_price:
                self._candidate_price = price
                self._candidate_index = idx
            else:
                drop = (self._candidate_price - price) / self._candidate_price
                if drop >= self.reversal_pct:
                    # Confirm HIGH pivot
                    self._pivots.append(
                        Pivot(
                            index=self._candidate_index,
                            time=candle.timestamp,
                            price=self._candidate_price,
                            kind="high",
                        )
                    )

                    self._direction = "down"
                    self._last_pivot_price = self._candidate_price
                    self._last_pivot_index = self._candidate_index

                    self._candidate_price = price
                    self._candidate_index = idx

        # ----------------------------------------------------
        # DOWN TREND
        # ----------------------------------------------------
        elif self._direction == "down":
            if price < self._candidate_price:
                self._candidate_price = price
                self._candidate_index = idx
            else:
                rise = (price - self._candidate_price) / self._candidate_price
                if rise >= self.reversal_pct:
                    # Confirm LOW pivot
                    self._pivots.append(
                        Pivot(
                            index=self._candidate_index,
                            time=candle.timestamp,
                            price=self._candidate_price,
                            kind="low",
                        )
                    )

                    self._direction = "up"
                    self._last_pivot_price = self._candidate_price
                    self._last_pivot_index = self._candidate_index

                    self._candidate_price = price
                    self._candidate_index = idx

    # --------------------------------------------------------

    def value(self):
        """
        Return all confirmed pivots.
        """
        return self._pivots

    def pivots(self) -> List[Pivot]:
        return self._pivots

    def last_pivot(self) -> Optional[Pivot]:
        if not self._pivots:
            return None
        return self._pivots[-1]

    def ready(self) -> bool:
        return len(self._pivots) > 0

    # --------------------------------------------------------
    # State persistence
    # --------------------------------------------------------

    def get_state(self):
        return {
            "candles_seen": self._candles_seen,
            "direction": self._direction,
            "last_pivot_price": self._last_pivot_price,
            "last_pivot_index": self._last_pivot_index,
            "candidate_price": self._candidate_price,
            "candidate_index": self._candidate_index,
            "pivots": [
                {
                    "index": p.index,
                    "time": p.time,
                    "price": p.price,
                    "kind": p.kind,
                }
                for p in self._pivots
            ],
        }

    def set_state(self, state: dict):
        self._candles_seen = state.get("candles_seen", 0)
        self._direction = state.get("direction")
        self._last_pivot_price = state.get("last_pivot_price")
        self._last_pivot_index = state.get("last_pivot_index")
        self._candidate_price = state.get("candidate_price")
        self._candidate_index = state.get("candidate_index")

        self._pivots = [
            Pivot(**p) for p in state.get("pivots", [])
        ]


In [ ]:
import backtrader as bt

from backtrader_extended.strategies.mbs.MBS import MBS


class After_MBS(MBS):
    pass


# Run One Example

In [ ]:
# sizer_params = {"initial_buy_amount_factor":strategy_class.params.initial_buy_amount_factor,  # buggy, it reads default values of strategy_class
#                  "martingale_multiplier":strategy_class.params.martingale_multiplier,  # buggy, it reads default values of strategy_class
#                  "data_in_market_cap": mcap,
#                  "log":sizer_log}
# from strategies.Fibo78Once import FiboR78Once
mcap=True
log=True
sizer_log=True

# from strategies.SimpleMartingaleStrategy import MartingaleSizer, SimpleMartingaleStrategy
###########################################################################################
from backtrader_extended.strategies.mbs.MBS import MBS,After_MBS
strategy_class = After_MBS
strategy_params = {
    # 'data_in_market_cap': mcap,
    #                 "log": log,
    #                 'tp': 1.2,                # Take profit at 120% of avg price
    #                 'sl': 0.7,
    #                 'buy_again': 0.8,         # Buy again at 70% of avg/last price
    #                 'max_buy_count': 2,
    #                 'end_mcap': 20_000,
    #                 'min_ib_mcap': 40_000 ,
    #                 "sell_on_no_loss": False ,
    #                 #
    #                 'uncatch_bounce_tp': 1.2 ,
    #                 "sell_on_current_bounce_from_min": 1.25 ,
    #                 "sell_on_no_loss": True ,
    #                 "up_bounce_threshold": 1.1 ,
    #                 "down_bounce_threshold": 0.9 ,
    #                 "no_buy_on_down_fall": True ,
    #                 "min_liveliness_for_ba": 0.5 ,
    #                "add_dynamic_ba":0.05,

    #                 'dead_coin_market_cap': 9_000,
    #                 'migration_market_cap': 125_000,
    #                 'buy_again_avg': 0,   # Buy again when avg/last is less than buy_again of current price 0 = less
    #                 'sell_tp_on_avg': 0,  # sell tp on avg/last 0= doublemore
    #                 'sell_sl_on_avg': 0,  # sell tp on avg/last 0= less

                    }

###########################################################################################
choose_sizer="m"
cash = 100

if choose_sizer=="m":
  sizer_class= MartingaleSizer
  sizer_params={"stake_cash":20 * 10_000_000_000
                ,"multiplier":1.5
                ,"max_multiplier":32
                ,"percentage":2
                }

 # sizer_class = MartingaleSizer
# sizer_params={"stake":10000}
# sizer_params = {
#     "initial_buy_amount_factor": strategy_params.get('initial_buy_amount_factor', 0.05),
#     "martingale_multiplier": strategy_params.get('martingale_multiplier', 2.0),
#     "data_in_market_cap": mcap,
#     "log": sizer_log
# }
elif choose_sizer=="p":
  sizer_class = bt.sizers.PercentSizer
  sizer_params={'percents':20}
  # sizer_params = {"stake_cash":20*1_000_000_000}
# sizer_params = None

elif choose_sizer=="f":
  sizer_class = bt.sizers.FixedSize()
  sizer_params = {"stake":20 * 1_000_000_000}

##################
# sizer_class = Fibo78Once
# sizer_params = {
#     "initial_buy_amount_factor": strategy_params.get('initial_buy_amount_factor', 0.05),
#     "martingale_multiplier": strategy_params.get('martingale_multiplier', 2.0),
#     "data_in_market_cap": mcap,
#     "log": sizer_log
# }
###########################################################################################

# 216


@dataclass
class RunConfig:
    results_folder: str = '/content/drive/MyDrive/charts/results/'
    cash: float = 100
    mcap: bool = True
    after_ath: bool = False
    min_start_minutes_to_wait: int = 30
    randomize_start_margin: bool = True
    df_end_margin: int = -1
    max_start_margin: int = 100
    min_start_margin: int = 10
    cerebro_runonce: bool = False
config = RunConfig()

strategy_class = After_MBS
# strategy_class = MBS

index_start = 220
index_end = index_start + 1

all_results_df, all_cerebros_objects, all_portfolio_histories = run_all(csv_files_1s[index_start:index_end],
                                                                        sizer_class=sizer_class,
                                                                        strategy_class=strategy_class,
                                                                        strategy_params=strategy_params,
                                                                        sizer_params=sizer_params,
                                                                        config =config,
                                                                        )

log=False
sizer_log=False
# strategy_params={'data_in_market_cap': mcap,"log":log, 'buy_mcap':12_000,'tp_mcap':19_000}
strategy_params['data_in_market_cap'] = mcap
strategy_params["log"] = log
sizer_params["log"]=sizer_log
print(analys(dfname="name",df=all_results_df))
all_results_df

In [ ]:

d = pd.read_csv(results_folder + "BaseBuySell20_30_MartingaleSizer_2025-10-20_14:04:35_len-306_memes.csv")

main_list_str = d["main_list"].values[1]  # string from CSV
main_list = ast.literal_eval(main_list_str)  # convert to actual list of tuples

df = analyse_main_list(main_list)

df


# Run

In [ ]:
e

In [ ]:
# 1 , 2   ()       , 3              , 4              , 5              , 6
# 1 , 1.5(2.5=40%) , 2.25(4.75=21%) , 3.37 (8.1=12%) , 5  (13.1=7.5%) , 7.5 (20.5=4.8%)
# 1 , 2   (3=33%)  , 4   (7=14%)    , 8    (15=6.5%) , 16 (31=3.2%)   , 32  (63=1.5%) , 32
# min of percentage sizer for 1 coin. half for 2. 1/5 for five.
# Sizer 1.5,2 = 40% (meaning 40 + 1.5*40 = 100%), for 2=20% ,  , for 5=8%
# If min is 0.002 and All is 0.1 -> 2% you can go for 5 token risk and sizer1.5,4
# If min is 0.002 and All is 0.1 -> 2% you can go for 5 token risk and sizer2,3
# If min is 0.002 and All is 0.1 -> 2% you can go for 4 token risk and sizer2,4

#
# Sizer 1.5,2 = 40%   , for2= 20% , for3= 13% , for4= 10% , for5= 8%      # -
# Sizer 1.5,3 = 20%   , for2= 10% , for3= 7%  , for4= 5%  , for5= 4%      # -
# Sizer 1.5,4 = 12%   , for2= 6%  , for3= 4%  , for4= 3%  , for5= 2.4%    # -
# Sizer 1.5,5 = 7.5%  , for2= 3.7%, for3= 2.5%, for4= 1.8%, for5= 1.5%    # -
# Sizer 1.5,6 = 4.8%  , for2= 2.4%, for3= 1.6%, for4= 1.2%, for5= 0.9%    # -
#
# Sizer 2,2  = 33%   , for2= 16% , for3= 11% , for4= 8%  , for5= 6.5%    # -
# Sizer 2,3  = 14%   , for2= 7%  , for3= 4.5%, for4= 3.5%, for5= 2.8%    # -
# Sizer 2,4  = 6.5%  , for2= 3.2%, for3= 2.1%, for4= 1.6%, for5= 1.3%    # -
# Sizer 2,5  = 3.2%  , for2= 1.6%, for3= 1.1%, for4= 0.8%, for5= 0.6%    # -
# Sizer 2,6  = 1.5%  , for2= 0.7%, for3= 0.5%, for4= 0.3%, for5= 0.3%    # -




In [ ]:
# 1 , 2   ()       , 3              , 4              , 5              , 6
# 1 , 1.5(2.5=40%) , 2.25(4.75=21%) , 3.37 (8.1=12%) , 5  (13.1=7.5%) , 7.5 (20.5=4.8%)
# 1 , 2   (3=33%)  , 4   (7=14%)    , 8    (15=6.5%) , 16 (31=3.2%)   , 32  (63=1.5%) , 32
# min of percentage sizer for 1 coin. half for 2. 1/5 for five.
# Sizer 1.5,2 = 40% (meaning 40 + 1.5*40 = 100%), for 2=20% ,  , for 5=8%
# If min is 0.002 and All is 0.1 -> 2% you can go for 5 token risk and sizer1.5,4
# If min is 0.002 and All is 0.1 -> 2% you can go for 5 token risk and sizer2,3
# If min is 0.002 and All is 0.1 -> 2% you can go for 4 token risk and sizer2,4

#
# Sizer 1.5,2 = 40%   , for2= 20% , for3= 13% , for4= 10% , for5= 8%      # -
# Sizer 1.5,3 = 20%   , for2= 10% , for3= 7%  , for4= 5%  , for5= 4%      # -
# Sizer 1.5,4 = 12%   , for2= 6%  , for3= 4%  , for4= 3%  , for5= 2.4%    # -
# Sizer 1.5,5 = 7.5%  , for2= 3.7%, for3= 2.5%, for4= 1.8%, for5= 1.5%    # -
# Sizer 1.5,6 = 4.8%  , for2= 2.4%, for3= 1.6%, for4= 1.2%, for5= 0.9%    # -
#
# Sizer 2,2  = 33%   , for2= 16% , for3= 11% , for4= 8%  , for5= 6.5%    # -
# Sizer 2,3  = 14%   , for2= 7%  , for3= 4.5%, for4= 3.5%, for5= 2.8%    # -
# Sizer 2,4  = 6.5%  , for2= 3.2%, for3= 2.1%, for4= 1.6%, for5= 1.3%    # -
# Sizer 2,5  = 3.2%  , for2= 1.6%, for3= 1.1%, for4= 0.8%, for5= 0.6%    # -
# Sizer 2,6  = 1.5%  , for2= 0.7%, for3= 0.5%, for4= 0.3%, for5= 0.3%    # -
zp={"stake_cash":20 * 1_000_000_000
                ,"multiplier": 1.5
                ,"max_multiplier":32
                ,"percentage": 20
                ,"log":False
                }
sp={'data_in_market_cap': mcap,
            "log": False,
            'tp': 1.2,                # Take profit at 120% of avg price
            'sl': 0.5,
            'buy_again': 0.8,         # Buy again at 70% of avg/last price
            'max_buy_count': 5,
            'min_ib_mcap': 30_000,
            'end_mcap': 15_000,
            'buy_again_from_top': False,
            # All False 1.098 1.088
            # TTT  1.08
            # TTF  1.07
            # FTT  1.12 best!
            # TFT  1.05
            #
            'TEMA_p': 150,
            'DEMA_p': 150,
            'use_dema': [False, False, False, False, False],
            'use_tema': [False, False, False, False, False],
            #
            'use_macd': [False, False, False, False, False],   # only this 1.055(20) 1.075(30)   (60)
            'use_ema': [False, False, False, False, False],   # only this 1.046(0.1)  0.996(0.05) 1.077(0.2) 1.11(0.3) 1.1(0.4)
            'use_rsi': [False, False, False, False, False],   # only this 1.086(50-p30) 1.07(30-p30) ========  1.026(60-p60) 1.126(50-p60) 1.076(40-p60)  1.012(30-p60)
            'use_liveliness': [False, False, False, False, False], #
            'use_stochastic': [False, False, False, False, False], #
            'use_bounce': [False, False, False, False, False], #
            'use_down_fall': [False, False, False, False, False], #
            'MACD_p': [30, 65, 23],
                # ('use_indicator', [initial_buy, buy_again, sell_tp, sell_sl, sell_custom]),
            # 'MACD_p': [20, 44, 16],
            # 'MACD_2X': [30, 65, 23],
            # 'MACD_5X': [60, 130, 50],
            'EMA_p': 60,
            'RSI_p': 60,
            'STOCH_p': [60, 15, 15],
            "liveliness_w": 120,
            "rsi_thr": 50,
            "EMA_tolerance": 0.3,

            'uncatch_bounce_tp':  1.2,
            "sell_on_current_bounce_from_min" : 2, # 2 was best! 1.4 was best
            "bounce_threshold" :   [1.3, 0.7],
            "buy_after_bounce_down" : [-0.3, 1.1, "down"],

            # "no_buy_on_down_fall": False, # best was False!
            #
            "min_liveliness_for_ba": 0 ,
            "add_dynamic_ba":0.00, # best was 0 !

            "sell_on_no_loss": False ,
            'dead_coin_market_cap': 9_000,
            'migration_market_cap': 425_000,
            'buy_again_avg': 0,   # Buy again when avg/last is less than buy_again of current price 0 = less
            'sell_tp_on_avg': 0,  # sell tp on avg/last 0= doublemore
            'sell_sl_on_avg': 0,  # sell tp on avg/last 0= less
            }





@dataclass
class RunConfig:
    results_folder: str = '/content/drive/MyDrive/charts/results/'
    cash: float = 100
    mcap: bool = True
    after_ath: bool = False
    min_start_minutes_to_wait: int = 20
    randomize_start_margin: bool = True
    df_end_margin: int = -1
    max_start_margin: int = 60
    min_start_margin: int = 10
    cerebro_runonce: bool = False

config = RunConfig()
#  6 400
# 78910 400 900

list_of_files_to_run = new_csv_files_1s
list_of_files_to_run = csv_files_1s[-50:] #
is_there_duplicate = False
# is_there_duplicate = if_duplicate(file_to_run=list_of_files_to_run,
#             sizer_class=sizer_class,
#             strategy_class=strategy_class,
#             strategy_params=sp
#             , sizer_params=zp
#             , config=config)

if not is_there_duplicate:


    sp["bounce_threshold"] = [1.2, 0.8]
    sp["buy_after_bounce_down"] = [-0.3, 1.1, "down"]

    strategy_class = After_MBS
    config.after_ath= True if strategy_class == After_MBS else False
    all_results_df, all_cerebros_objects, all_portfolio_histories=run_and_save(file_to_run=list_of_files_to_run ,
                sizer_class=sizer_class,
                strategy_class=strategy_class,
                strategy_params=sp
                , sizer_params=zp
                , config=config )

    sp["bounce_threshold"] = [1.2, 0.8]
    sp["buy_after_bounce_down"] = [-0.3, 1.1, "up"]


    strategy_class = After_MBS
    config.after_ath= True if strategy_class == After_MBS else False
    all_results_df, all_cerebros_objects, all_portfolio_histories=run_and_save(file_to_run=list_of_files_to_run ,
                sizer_class=sizer_class,
                strategy_class=strategy_class,
                strategy_params=sp
                , sizer_params=zp
                , config=config )


    # strategy_class = BaseBuySell20_30_INDICATORS
    # config.after_ath= True if strategy_class == After_BaseBuySell20_30_INDICATORS else False

    # run_and_save(file_to_run=list_of_files_to_run,
    #             sizer_class=sizer_class,
    #             strategy_class=strategy_class,
    #             strategy_params=sp
    #             , sizer_params=zp
    #             , config=config)







In [ ]:
fil = all_results_df[all_results_df["final_value"]!=1]
print(fil["final_value"].sum() / len(fil["final_value"]))
print(fil["final_value"].mean())

In [ ]:
all_results_df

In [ ]:
# pd.read_csv("/content/drive/MyDrive/charts/results/After_BaseBuySell20_30_INDICATORS_MartingaleSizer_2025-12-16_15:18:40_len-3_memes.csv").columns

In [ ]:

for f in list_of_files_to_run:
    print(f)
    print(pd.read_csv(f).tail())
# all_results_df

In [ ]:
# !cp "/content/drive/MyDrive/charts/1s/axiom_chart_bars_2hpLgGFKAGPkxUj7Hu1QXL1DnhxVDNJbjMSgNa15UNEc_1762016631330.csv" foo.txt.tmp
# !sed '$ d' foo.txt.tmp > foo.txt
# !rm -f  "/content/drive/MyDrive/charts/1s/axiom_chart_bars_2hpLgGFKAGPkxUj7Hu1QXL1DnhxVDNJbjMSgNa15UNEc_1762016631330.csv"
# !cp foo.txt "/content/drive/MyDrive/charts/1s/axiom_chart_bars_2hpLgGFKAGPkxUj7Hu1QXL1DnhxVDNJbjMSgNa15UNEc_1762016631330.csv"

In [ ]:
# tmpdf = pd.read_csv("/content/drive/MyDrive/charts/results/BaseBuySell20_30_MartingaleSizer_2025-10-16_12:14:00_len-306_memes.csv")
# print(tmpdf.columns)
# tmpdf = tmpdf[["coin","ath","time_token","time_to_ath",	"time_after_ath", 'len_index_token',       'index_ath']]
# tmpdf.to_csv(results_folder + "306_ath.csv")
# tmpdf

# **Results**

In [ ]:
import pandas as pd
import os
df_306_ath = pd.read_csv("/content/drive/MyDrive/charts/results/306_ath.csv")
# Main analysis loop - collect all results into a DataFrame
print(df_306_ath.head())
results_list = []
print("=" * 80)

results_files_list = []

for f in os.listdir(results_folder):
    if f.endswith('.csv') and not f.startswith("all_portfolio_histories") and not f.endswith("_ath.csv") and f.endswith("_memes.csv"):
      results_files_list.append(f)

print(len(results_files_list))
for f in results_files_list[:]:
      lenstr=f.split("_")[-2].split("-")[-1]
      if not f.startswith("After_MBS"):
        continue
      if int(lenstr)>40:
        print(f)
        result = analys(os.path.join(results_folder, f),ath_df=df_306_ath)
        print(result)
        # e/100
        results_list.append(result)

# Create comprehensive results DataFrame
summary_df = pd.DataFrame(results_list)

# Display the results
print("Analysis Summary:")
print("=" * 80)
print(summary_df.columns)
print(summary_df.to_string(index=False))

# Optional: Save to CSV
# summary_df.to_csv('trading_analysis_summary.csv', index=False)

# # You can also access specific columns or filter the data
# print("\n" + "=" * 80)
# print("Top 5 Most Profitable (by PnL %):")
# top_profitable = summary_df.nlargest(5, 'pnl_percentage')[['filename', 'total_trades', 'profitable_percentage', 'pnl_percentage']]
# print(top_profitable.to_string(index=False))

# print("\n" + "=" * 80)
# print("Summary Statistics:")
# print(f"Total files analyzed: {len(summary_df)}")
# print(f"Average profitable percentage: {summary_df['profitable_percentage'].mean():.2f}%")
# print(f"Average PnL percentage: {summary_df['pnl_percentage'].mean():.2f}%")
# print(f"Files with errors: {len(summary_df[summary_df['total_trades'] == 0])}")

In [ ]:
summary_df.to_csv('/content/drive/MyDrive/charts/trading_analysis_summary.csv', index=False)


In [ ]:
# print(summary_df.to_string(index=False))
print(summary_df.to_string(index=False))


In [ ]:
print(summary_df.sort_values(["geo_mean_return"]).to_string(index=False))



In [ ]:
tmpdf=summary_df[summary_df["filename"].str.startswith("After_")].sort_values("%total_pnl_for_changed")

print(tmpdf.to_string(index=False))

In [ ]:
t=merged_df[merged_df["filename"]=="After_BaseBuySell20_30_MartingaleSizer_2025-10-15_08:55:20_len-306_memes.csv"]
print(t.to_string(index=False))


# Result of One

In [ ]:
name = "After_MBS_MartingaleSizer_2026-02-19_00:11:53_len-50"
filename =name + "_memes.csv"
details_file = "details_" + name + "_details.txt"
portfolio_file ="all_portfolio_histories" + name

this_run_results = pd.read_csv(results_folder + filename).sort_values("final_value")
this_run_results[this_run_results["final_value"]>2]

In [ ]:
this_run_results[this_run_results["coin"]=="61V5Q9rUVy"]["main_list"].values

candle_file = "/content/drive/MyDrive/charts/1s" + "axiom_chart_bars_61V5Q9rUVygarHA8VceBePTDUyhrcWgZ1ACdNyeb5XG2_1755001592579.csv"
candle_file
candles_folder =  "/content/drive/MyDrive/charts/1s/"
candle_file = candles_folder + "axiom_chart_bars_9ks6HyRrmifkR8racHoU8G5Ta6fTYDQXz8qVShm18PiY_1755004233760.csv"
# candle_file = candles_folder  + "axiom_chart_bars_61V5Q9rUVygarHA8VceBePTDUyhrcWgZ1ACdNyeb5XG2_1755001592579.csv"
# coin = "61V5Q9rUVy"
coin = "9ks6HyRrmi"

df = pd.read_csv(candle_file)
df

In [ ]:
!ls "/content/drive/MyDrive/charts/1s" | grep 61V5Q9rUVy


In [ ]:
file_path= results_folder + portfolio_file

with open(file_path, 'rb') as f: # 'rb' for read binary
    print("openning ",file_path)
    all_portfolio_histories = pickle.load(f)
print(f"\nDictionary successfully loaded from all_portfolio_histories")
print("Loaded dictionary:",len(all_portfolio_histories))

In [ ]:
# pd.read_csv("/content/drive/MyDrive/charts/1s/axiom_chart_bars_6dPC9QinSCNgoapREEP7fyx9jgsTqgpGJJ841HRf7Tr9_1762011668172.csv").tail()
# pd.read_csv("/content/drive/MyDrive/charts/1s/axiom_chart_bars_61V5Q9rUVygarHA8VceBePTDUyhrcWgZ1ACdNyeb5XG2_1755001592579.csv").tail()
# this_run_results[this_run_results["coin"]=="5kM3jSj5aN"]["main_list"].values[0]

coin_1s_csv =  "axiom_chart_bars_9ks6HyRrmifkR8racHoU8G5Ta6fTYDQXz8qVShm18PiY_1755004233760.csv"
pd.read_csv("/content/drive/MyDrive/charts/1s/"+coin_1s_csv).head()
pd.read_csv(new_csv_files_1s[3]).head()

In [ ]:
draw_a_coin_from_results(this_run_results,all_portfolio_histories,0)

In [ ]:
from bokeh.palettes import Category10
from bokeh.models import (
    ColumnDataSource, HoverTool, CrosshairTool, Span,
    DatetimeTickFormatter, NumeralTickFormatter, BoxAnnotation
)
from bokeh.layouts import column
from bokeh.plotting import figure, show, output_notebook
from ast import literal_eval
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import os


def draw_a_coin_from_results(this_run_results, all_portfolio_histories, index=None, coin_name=None, extra_indicators=None):
    if index is not None:
        main_list_str = this_run_results.iloc[index]["main_list"]
        coin_name = this_run_results.iloc[index]["coin"]
    elif coin_name is not None:
        main_list_str = this_run_results[this_run_results["coin"] == coin_name]["main_list"].values[0]
        coin_name = coin_name
    else:
        print("Coin and index is None!")
        return
    print("drawing for ", coin_name)
    main_list = literal_eval(main_list_str)
    draw_zoomable_chart(main_list, coin=coin_name, filename=find_full_candle_file_name(coin_name), portfolio_series=all_portfolio_histories[coin_name], show_candles=True, extra_indicators=extra_indicators)


def find_full_candle_file_name(coin, dir="/content/drive/MyDrive/charts/1s/"):
    for f in os.listdir(dir):
        if coin in f:
            return dir + f
    print(coin, "Not found! in", dir)


def draw_zoomable_chart(main_list, coin, filename, portfolio_series=None, show_candles=False, extra_indicators=None):
    # === Load candle data ===
    df = pd.read_csv(filename)
    if "time" in df.columns:
        df["timestamp"] = pd.to_datetime(df["time"], unit="ms", errors="coerce")
    else:
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms", errors="coerce")
    df = df.sort_values("time")
    df = df.dropna(subset=["open", "high", "low", "close"])
    df[["open", "high", "low", "close"]] = df[["open", "high", "low", "close"]].astype(float)
    df[["open", "high", "low", "close"]] = df[["open", "high", "low", "close"]] * 1_000_000_000

    # === Compute average volume ===
    vol_col = None
    for candidate in ["VolumeTradedinbaseasset", "volume", "Volume"]:
        if candidate in df.columns:
            vol_col = candidate
            break
    if vol_col is not None:
        df["adj_volume"] = df[vol_col] * 1_000_000 / df["close"]
        df["avg_volume_60"] = df["adj_volume"].rolling(60, min_periods=1).mean()
    else:
        print("[Warning] No volume column found — skipping avg_volume_60")
        df["avg_volume_60"] = 0

    # === Extract trade events ===
    events = pd.DataFrame(main_list, columns=["type", "price", "index", "time", "min", "max"])
    events["time"] = pd.to_datetime(events["time"])

    # === Define colors ===
    colors = {"ib": "white", "ba": "orange", "tp": "lime", "sl": "red", "e": "red", "nl": "orange", "bfm": "green", "d": "red"}
    for k, v in colors.items():
        print(k, len(events[events["type"] == k]))

    # === Dynamic subplot layout ===
    has_indicators = bool(extra_indicators)
    has_portfolio = portfolio_series is not None

    rows = 1
    row_heights = [0.55]
    subtitles = [f"{coin} — Trade Events"]
    specs = [[{"secondary_y": True}]]

    portfolio_row = None
    indicator_rows = []

    for ind in (extra_indicators or []):
        rows += 1
        indicator_rows.append(rows)
        row_heights.append(0.2)
        subtitles.append(ind["name"])
        specs.append([{}])
    # if has_indicators:
    #     rows += 1
    #     indicator_row = rows
    #     row_heights.append(0.25)
    #     subtitles.append(", ".join(ind["name"] for ind in extra_indicators))
    #     specs.append([{}])

    if has_portfolio:
        rows += 1
        portfolio_row = rows
        row_heights.append(0.2)
        subtitles.append("Portfolio Value Over Time")
        specs.append([{}])

    total = sum(row_heights)
    row_heights = [h / total for h in row_heights]

    fig = make_subplots(
        rows=rows, cols=1,
        shared_xaxes=True,
        row_heights=row_heights,
        vertical_spacing=0.05,
        subplot_titles=subtitles,
        specs=specs
    )

    # === Candlestick chart ===
    if show_candles:
        fig.add_trace(go.Candlestick(
            x=df["timestamp"],
            open=df["open"], high=df["high"], low=df["low"], close=df["close"],
            name="Candles",
            increasing_line_color="rgba(0,255,0,0.3)", decreasing_line_color="rgba(255,0,0,0.3)",
            increasing_fillcolor="rgba(0,255,0,0.2)", decreasing_fillcolor="rgba(255,0,0,0.2)",
            showlegend=False
        ), row=1, col=1, secondary_y=False)

    # === Avg volume ===
    fig.add_trace(go.Scatter(
        x=df["timestamp"], y=df["avg_volume_60"],
        mode="lines", name="Avg Volume (60)",
        line=dict(color="gold", width=1.5, dash="dot"), opacity=0.7
    ), row=1, col=1, secondary_y=True)

    # === Trade event markers ===
    for t, c in colors.items():
        sub = events[events["type"] == t]
        if not sub.empty:
            fig.add_trace(go.Scatter(
                x=sub["time"],  # ← correct: event timestamps, not df timestamps
                y=sub["price"] * 1.1 if t in ["ib"] else sub["price"],
                mode="markers+text",
                text=sub["type"],
                textposition="top center",
                textfont=dict(size=9, color=c),
                marker=dict(color=c, size=9 if t in ["ib"] else 8,
                            opacity=0.8, line=dict(width=1, color="black")),
                name=t.upper(), showlegend=True
            ), row=1, col=1, secondary_y=False)

    # === Indicators ===
    # if has_indicators:
    #     for ind in extra_indicators:
    #         fig.add_trace(go.Scatter(
    #             x=ind.get("x", df["timestamp"]),
    #             y=ind["series"],
    #             mode="lines", name=ind["name"],
    #             line=dict(color=ind.get("color", "white"), width=1.5)
    #         ), row=indicator_row, col=1)
    #         for level in ind.get("hlines", []):
    #             fig.add_hline(y=level, line_dash="dot", line_color="gray", row=indicator_row, col=1)
    if has_indicators:
        for ind, ind_row in zip(extra_indicators, indicator_rows):
            fig.add_trace(go.Scatter(
                x=ind.get("x", df["timestamp"]),
                y=ind["series"],
                mode="lines", name=ind["name"],
                line=dict(color=ind.get("color", "white"), width=1.5)
            ), row=ind_row, col=1)
            for level in ind.get("hlines", []):
                fig.add_hline(y=level, line_dash="dot", line_color="gray", row=ind_row, col=1)
    # === Portfolio ===
    if has_portfolio:
        portfolio_series.index = pd.to_datetime(portfolio_series.index)
        fig.add_trace(go.Scatter(
            x=portfolio_series.index, y=portfolio_series.values,
            mode="lines+markers", name="Portfolio Value",
            line=dict(color="cyan", width=2), marker=dict(size=4)
        ), row=portfolio_row, col=1)
        fig.update_yaxes(type="log", row=portfolio_row, col=1)  # ← fixed, was hardcoded row=3

    # === Layout ===
    fig.update_layout(
        uirevision="constant",
        template="plotly_dark",
        hovermode="x unified",
        xaxis_rangeslider_visible=False,
        height=950,
        legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="right", x=1)
    )
    # Set every y-axis: autorange=True, fixedrange=False
    for i in range(1, rows + 1):
        yaxis_name = f"yaxis{i}" if i > 1 else "yaxis"
        fig.update_layout(**{yaxis_name: dict(autorange=True, fixedrange=False)})
    # Also do the secondary y on row 1
    fig.update_layout(yaxis2=dict(autorange=True, fixedrange=False))

    fig.update_yaxes(title_text="Price", secondary_y=False, row=1, col=1)
    fig.update_yaxes(title_text="Avg Volume (60)", secondary_y=True, row=1, col=1)

    fig.show()
# df["RSI_14"] = compute_rsi(df["close"], 14)
# df["RSI_30"] = compute_rsi(df["close"], 30)

# extra_indicators=[
#         {"name": "RSI-14", "series": df["RSI_14"].values, "color": "deepskyblue", "hlines": [30, 70]},
#         {"name": "RSI-30", "series": df["RSI_30"].values, "color": "deepskyblue", "hlines": [30, 70]},
# ]
# Don't pass .values — pass the Series with its index intact, or pass x explicitly
# extra_indicators=[
#     {"name": "RSI-14", "series": df["RSI_14"], "x": df["time"], "color": "deepskyblue", "hlines": [30, 70]},
#     {"name": "RSI-30", "series": df["RSI_30"], "x": df["time"], "color": "orange",      "hlines": [30, 70]},
# ]


output_notebook()


def draw_a_coin_from_results(this_run_results, all_portfolio_histories, index=None, coin_name=None, extra_indicators=None):
    if index is not None:
        main_list_str = this_run_results.iloc[index]["main_list"]
        coin_name = this_run_results.iloc[index]["coin"]
    elif coin_name is not None:
        main_list_str = this_run_results[this_run_results["coin"] == coin_name]["main_list"].values[0]
    else:
        print("Coin and index is None!")
        return
    print("drawing for", coin_name)
    main_list = literal_eval(main_list_str)
    draw_zoomable_chart(
        main_list, coin=coin_name,
        filename=find_full_candle_file_name(coin_name),
        portfolio_series=all_portfolio_histories[coin_name],
        show_candles=True,
        extra_indicators=extra_indicators
    )


def find_full_candle_file_name(coin, dir="/content/drive/MyDrive/charts/1s/"):
    for f in os.listdir(dir):
        if coin in f:
            return dir + f
    print(coin, "Not found! in", dir)


def draw_zoomable_chart(main_list, coin, filename, portfolio_series=None, show_candles=False, extra_indicators=None):
    # === Load candle data ===
    df = pd.read_csv(filename)
    if "time" in df.columns:
        df["timestamp"] = pd.to_datetime(df["time"], unit="ms", errors="coerce")
    else:
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms", errors="coerce")
    df = df.sort_values("timestamp")
    df = df.dropna(subset=["open", "high", "low", "close"])
    df[["open", "high", "low", "close"]] = df[["open", "high", "low", "close"]].astype(float) * 1_000_000_000

    # === Resolve indicator series from functions ===
    resolved_indicators = []
    for ind in (extra_indicators or []):
        resolved = dict(ind)  # copy so we don't mutate the original
        if "fn" in resolved:
            resolved["series"] = resolved.pop("fn")(df)
        resolved.setdefault("x", df["timestamp"])
        resolved_indicators.append(resolved)
    has_indicators = bool(resolved_indicators)

    # === Volume ===
    vol_col = None
    for candidate in ["VolumeTradedinbaseasset", "volume", "Volume"]:
        if candidate in df.columns:
            vol_col = candidate
            break
    if vol_col is not None:
        df["adj_volume"] = df[vol_col] * 1_000_000 / df["close"]
        df["avg_volume_60"] = df["adj_volume"].rolling(60, min_periods=1).mean()
    else:
        df["avg_volume_60"] = 0

    # === Trade events ===
    events = pd.DataFrame(main_list, columns=["type", "price", "index", "time", "min", "max"])
    events["time"] = pd.to_datetime(events["time"])

    colors = {
        "ib": "white",
        "ba": "orange",
        "tp": "lime",
        "sl": "red",
        "e": "red",
        "nl": "orange",
        "bfm": "green",
        "d": "red"
    }
    for k in colors:
        print(k, len(events[events["type"] == k]))

    # === Shared x range (drives linked zoom) ===
    x_min = df["timestamp"].min()
    x_max = df["timestamp"].max()

    TOOLS = "xpan,ypan,xwheel_zoom,xbox_zoom,reset,save"
    DARK_BG = "#0f1117"
    GRID_COL = "#2a2a3a"
    TEXT_COL = "#cccccc"

    def base_fig(height, y_axis_label="", x_range=None):
        kwargs = dict(
            width=1400, height=height,
            background_fill_color=DARK_BG,
            border_fill_color=DARK_BG,
            outline_line_color=GRID_COL,
            tools=TOOLS,
            active_scroll="xwheel_zoom",
            x_axis_type="datetime",
            y_axis_label=y_axis_label,
        )
        if x_range is not None:
            kwargs["x_range"] = x_range
        p = figure(**kwargs)
        p.xgrid.grid_line_color = GRID_COL
        p.ygrid.grid_line_color = GRID_COL
        p.xaxis.major_label_text_color = TEXT_COL
        p.yaxis.major_label_text_color = TEXT_COL
        p.yaxis.axis_label_text_color = TEXT_COL
        p.xaxis.formatter = DatetimeTickFormatter(
            hours="%d %b %H:%M", days="%d %b", months="%b %Y"
        )
        p.add_tools(CrosshairTool(line_color="gray", line_alpha=0.5))
        return p

    # ── Row 1: Price ──────────────────────────────────────────────
    p_price = base_fig(500, y_axis_label="Price")
    p_price.title.text = f"{coin} — Trade Events"
    p_price.title.text_color = TEXT_COL

    if show_candles:
        inc = df["close"] >= df["open"]
        dec = ~inc
        w = 30_000  # candle width in ms — adjust to your candle interval

        src_inc = ColumnDataSource(dict(
            x=df["timestamp"][inc], top=df["close"][inc], bottom=df["open"][inc],
            high=df["high"][inc], low=df["low"][inc]
        ))
        src_dec = ColumnDataSource(dict(
            x=df["timestamp"][dec], top=df["open"][dec], bottom=df["close"][dec],
            high=df["high"][dec], low=df["low"][dec]
        ))
        p_price.segment("x", "high", "x", "low", source=src_inc, color="rgba(0,200,0,0.4)")
        p_price.segment("x", "high", "x", "low", source=src_dec, color="rgba(200,0,0,0.4)")
        p_price.vbar("x", w, "top", "bottom", source=src_inc,
                     fill_color="rgba(0,200,0,0.25)", line_color="rgba(0,200,0,0.5)")
        p_price.vbar("x", w, "top", "bottom", source=src_dec,
                     fill_color="rgba(200,0,0,0.25)", line_color="rgba(200,0,0,0.5)")
    else:
        p_price.line(df["timestamp"], df["close"], color="#4488ff", line_width=1.5, legend_label="Close")

    # Volume on secondary y — Bokeh doesn't have native secondary y, use extra_y_ranges
    from bokeh.models import LinearAxis, Range1d
    vol_max = df["avg_volume_60"].max()
    p_price.extra_y_ranges = {"vol": Range1d(start=0, end=vol_max * 4)}
    vol_axis = LinearAxis(y_range_name="vol", axis_label="Avg Vol (60)",
                          axis_label_text_color=TEXT_COL, major_label_text_color=TEXT_COL)
    p_price.add_layout(vol_axis, "right")
    p_price.line(df["timestamp"], df["avg_volume_60"],
                 color="gold", line_width=1.2, line_dash="dashed",
                 alpha=0.6, y_range_name="vol", legend_label="Avg Vol 60")

    # Trade event markers
    for t, c in colors.items():
        sub = events[events["type"] == t]
        if sub.empty:
            continue
        y = sub["price"] * 1.1 if t == "ib" else sub["price"]
        src = ColumnDataSource(dict(x=sub["time"], y=y, label=sub["type"]))
        p_price.scatter("x", "y", source=src, color=c, size=9 if t == "ib" else 7,
                        alpha=0.85, marker="circle", legend_label=t.upper())
        p_price.add_tools(HoverTool(renderers=[
            p_price.scatter("x", "y", source=src, color=c, size=0, alpha=0)
        ], tooltips=[("Type", "@label"), ("Price", "@y{0.00000000}"), ("Time", "@x{%F %T}")],
            formatters={"@x": "datetime"}, mode="mouse"))

    p_price.legend.background_fill_color = "#1a1a2a"
    p_price.legend.label_text_color = TEXT_COL
    p_price.legend.border_line_color = GRID_COL
    p_price.legend.click_policy = "hide"

    panels = [p_price]

    # ── Indicator rows ────────────────────────────────────────────
    palette = Category10[10]
    for i, ind in enumerate(resolved_indicators or []):
        p_ind = base_fig(180, y_axis_label=ind["name"], x_range=p_price.x_range)
        x = ind.get("x", df["timestamp"])
        p_ind.line(x, ind["series"], color=ind.get("color", palette[i % 10]),
                   line_width=1.5, legend_label=ind["name"])

        for level in ind.get("hlines", []):
            hline = Span(location=level, dimension="width",
                         line_color="gray", line_dash="dashed", line_width=1, line_alpha=0.6)
            p_ind.add_layout(hline)

        p_ind.legend.background_fill_color = "#1a1a2a"
        p_ind.legend.label_text_color = TEXT_COL
        p_ind.legend.border_line_color = GRID_COL
        panels.append(p_ind)

    # ── Portfolio row ─────────────────────────────────────────────
    if portfolio_series is not None:
        portfolio_series.index = pd.to_datetime(portfolio_series.index)
        p_port = base_fig(180, y_axis_label="Portfolio", x_range=p_price.x_range)
        p_port.line(portfolio_series.index, portfolio_series.values,
                    color="cyan", line_width=1.8, legend_label="Portfolio Value")
        p_port.scatter(portfolio_series.index, portfolio_series.values,
                       color="cyan", size=3, alpha=0.5)
        p_port.yaxis.formatter = NumeralTickFormatter(format="0.0a")
        p_port.legend.background_fill_color = "#1a1a2a"
        p_port.legend.label_text_color = TEXT_COL
        p_port.legend.border_line_color = GRID_COL
        panels.append(p_port)

    # ── Render ────────────────────────────────────────────────────
    show(column(*panels, sizing_mode="stretch_width"))


# df["timestamp"] = pd.to_datetime(df["time"], unit="ms", errors="coerce")

# df["RSI_14"] = compute_rsi(df["close"], 14)
# df["ATR"] = compute_atr(df, 14)
extra_indicators = [
    {
        "name": "RSI-14",
        "fn": lambda df: compute_rsi(df["close"], 14),
        "color": "deepskyblue",
        "hlines": [30, 70]
    },
    {
        "name": "ATR",
        "fn": lambda df: compute_atr(df, 14),
        "color": "orange"
    },
    {
        "name": "NATR",
        "fn": lambda df: compute_natr_forman(df, 14),
        "color": "magenta",
        "hlines": [1, 3, 5]
    },
]


In [ ]:
from ast import literal_eval
import pandas as pd
import os
from bokeh.plotting import figure, show, output_notebook
from bokeh.layouts import column
from bokeh.models import (
    ColumnDataSource, HoverTool, CrosshairTool, Span,
    DatetimeTickFormatter, NumeralTickFormatter, BoxAnnotation
)
from bokeh.palettes import Category10


output_notebook()


df["timestamp"] = pd.to_datetime(df["time"], unit="ms", errors="coerce")

df["RSI_14"] = compute_rsi(df["close"], 14)
df["ATR"]    = compute_atr(df, 14)
extra_indicators=[
    {
        "name": "RSI-14",
        "fn": lambda df: compute_rsi(df["close"], 14),
        "color": "deepskyblue",
        "hlines": [30, 70]
    },
    {
        "name": "ATR",
        "fn": lambda df: compute_natr_normalized_first(df, 14),
        "color": "orange"
    },
    {
        "name": "NATR",
        "fn": lambda df: compute_natr_forman(df, 14),
        "color": "magenta",
        "hlines": [1, 3, 5]
    },
]
draw_a_coin_from_results(    this_run_results, all_portfolio_histories,    coin_name="2hpLgGFKAG",   extra_indicators= extra_indicators)

In [ ]:
coin_1s_csv =  "axiom_chart_bars_61V5Q9rUVygarHA8VceBePTDUyhrcWgZ1ACdNyeb5XG2_1755001592579.csv"
coin_1s_csv =  "axiom_chart_bars_9ks6HyRrmifkR8racHoU8G5Ta6fTYDQXz8qVShm18PiY_1755004233760.csv"
coin = coin_1s_csv[17:27]
draw_zoomable_chart(this_run_results,coin= coin,filename= candles_folder + coin_1s_csv , portfolio_series=all_portfolio_histories[coin],show_candles=True,extra_indicators=None)

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from ast import literal_eval

import pandas as pd
import plotly.graph_objects as go
from ast import literal_eval
from plotly.subplots import make_subplots
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def compute_rsi(series, period=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

def compute_atr(df, period=14):
    """Standard ATR in price units."""
    high = df["high"]
    low = df["low"]
    close = df["close"]

    prev_close = close.shift(1)
    tr = pd.concat([
        high - low,
        (high - prev_close).abs(),
        (low - prev_close).abs()
    ], axis=1).max(axis=1)

    return tr.rolling(window=period).mean()


def compute_natr_forman(df, period=14):
    """
    John Forman's NATR: ATR / Close * 100
    Order: true range → average → normalize
    """
    atr = compute_atr(df, period)
    return (atr / df["close"]) * 100


def compute_natr_normalized_first(df, period=14):
    """
    Alternative NATR: normalize true range first, then average.
    Order: true range → normalize → average
    Normalized True Range = Max(H-L, |H-C1|, |C1-L|) / C1
    """
    high = df["high"]
    low = df["low"]
    close = df["close"]
    prev_close = close.shift(1)

    normalized_tr = pd.concat([
        (high - low) / prev_close,
        (high - prev_close).abs() / prev_close,
        (low - prev_close).abs() / prev_close,
    ], axis=1).max(axis=1)

    return normalized_tr.rolling(window=period).mean() * 100
from utils.results_plot import draw_zoomable_chart , find_full_candle_file_name, draw_a_coin_from_results
df["RSI_14"] = compute_rsi(df["close"], 14)
df["RSI_30"] = compute_rsi(df["close"], 30)

# # extra_indicators=[
# #         {"name": "RSI-14", "series": df["RSI_14"].values, "color": "deepskyblue", "hlines": [30, 70]},
# #         {"name": "RSI-30", "series": df["RSI_30"].values, "color": "deepskyblue", "hlines": [30, 70]},
# # ]
# # Don't pass .values — pass the Series with its index intact, or pass x explicitly

df["ATR"]        = compute_atr(df, period=14)
df["NATR"]       = compute_natr_forman(df, period=14)
df["NATR_alt"]   = compute_natr_normalized_first(df, period=14)

extra_indicators = [
    {"name": "RSI-14", "series": df["RSI_14"], "x": df["time"], "color": "deepskyblue", "hlines": [30, 70]},
    # {"name": "RSI-30", "series": df["RSI_30"], "x": df["time"], "color": "orange",      "hlines": [30, 70]},

    {"name": "ATR",       "series": df["ATR"],      "x": df["time"], "color": "orange"},
    {"name": "NATR",      "series": df["NATR"],     "x": df["time"], "color": "magenta", "hlines": [1, 3, 5]},
    {"name": "NATR-alt",  "series": df["NATR_alt"], "x": df["time"], "color": "lime",    "hlines": [1, 3, 5]},
]

In [ ]:
df

# Plot Portfolios

In [ ]:
all_portfolio_histories["61V5Q9rUVy"]

In [ ]:
for k,v in all_portfolio_histories.items():
    keys,values = k,v
    print()


In [ ]:
import pandas as pd
s= values
# Suppose your series is called `s`
# Make sure index is datetime
s.index = pd.to_datetime(s.index)

# Find changes
changes = s[s != s.shift()]

# Add start, end, duration
df = pd.DataFrame({
    "value": changes.values,
    "start": changes.index,
})
df["end"] = df["start"].shift(-1).fillna(s.index[-1])
df["duration"] = df["end"] - df["start"]

print(df)


import pandas as pd

def get_timing_series(s):
    # Assuming your series is named s
    s.index = pd.to_datetime(s.index)

    # Find change points
    changes = s[s != s.shift()]

    df = pd.DataFrame({
        "value": changes.values,
        "start": changes.index,
    })
    df["end"] = df["start"].shift(-1).fillna(s.index[-1])
    df["duration"] = df["end"] - df["start"]

    # Break down duration
    df["duration_hours"] = df["duration"].dt.total_seconds() / 3600
    df["duration_minutes"] = df["duration"].dt.total_seconds() / 60
    df["duration_seconds"] = df["duration"].dt.total_seconds()

    print(df)
get_timing_series(s)


In [ ]:
file_path=results_folder + portfolio_file
# --- 3. Read the dictionary from the file using pickle ---
all_portfolio_histories = {}
try:
    with open(file_path, 'rb') as f: # 'rb' for read binary
        all_portfolio_histories = pickle.load(f)
    print(f"\nDictionary successfully loaded from all_portfolio_histories")
    print("Loaded dictionary:",len(all_portfolio_histories))
    # print(all_portfolio_histories)
except FileNotFoundError:
    print(f"Error: File not found at all_portfolio_histories")
except Exception as e:
    print(f"Error loading dictionary: {e}")

from utils.plotting_utils import plot_all_portfolio_histories_by_time,plot_all_portfolio_histories
plt.rcParams['figure.figsize'] = [18, 20] # Adjust as desired
plt.rcParams['figure.dpi'] = 100
# The %matplotlib inline is for Jupyter/Colab environment and should be run directly in a cell,
# For a script, you manage matplotlib output differently (e.g., plt.savefig)
%matplotlib inline

# Step 3: Plot all portfolio histories on one chart
plot_all_portfolio_histories_by_time(all_portfolio_histories, title="Portfolio Value Over Time for All Coins",legend=False)
plot_all_portfolio_histories(all_portfolio_histories, title="Portfolio Value Over Time for All Coins",legend=False)


In [ ]:
all_results_df

In [ ]:
all_cerebros_objects

In [ ]:
all_results_df

In [ ]:
all_cerebros_objects

In [ ]:

# Step 4: Select an interesting backtest and plot it
# You can choose based on 'sharpe_ratio', 'final_value', etc.
# For example, let's plot the coin with the highest final value
# best_coin_row = all_results_df.loc[all_results_df['final_value'].idxmax()]
best_coin_row = all_results_df.loc[0]

best_coin_name = best_coin_row['coin']
# best_coin_name="ars_ABPmWi"
print(f"\n--- Plotting best performing coin: {best_coin_name} ---")
if best_coin_name in all_cerebros_objects:
    plot_single_backtest(all_cerebros_objects[best_coin_name], title=f"Backtest for {best_coin_name}")
else:
    print(f"Cerebro object for {best_coin_name} not found.")

# You can also manually select a coin to plot, e.g.:
# plot_single_backtest(all_cerebros_objects['coin_0'], title="Backtest for coin_0")

# All

In [ ]:

details_df = read_all_detail_files()
details_df

In [ ]:

def merge_details_and_summary(summary_df, details_df):
    merged_df = pd.merge(
        summary_df,
        details_df,
        on="filename",   # join on common key
        how="inner"       # keep only matches
    )
    merged_df=merged_df[merged_df["total_tokens"]>10]
    print(merged_df.columns)
    return merged_df

merged_df = merge_details_and_summary(summary_df, details_df)
merged_df.sort_values(["filename","sizer_stake_cash"]).to_csv("/content/drive/MyDrive/charts/merged_df.csv")
merged_df

In [ ]:
sort_by_col = "%total_pnl_for_changed"
sort_by_col = "%profitable_tokens"
tmpdf=merged_df[merged_df["filename"].str.startswith("After_")].sort_values(sort_by_col, ascending=False)

print(tmpdf.to_string(index=False))

In [ ]:
print(merged_df[merged_df["geo_mean_return"]>110].sort_values("profitable_tokens").to_string(index=False))

In [ ]:
find_rows_with_same_att(269,merged_df,by_sizer=False,by_size=False,by_strategy_end=False)

In [ ]:
data_analys_list = []
for f in os.listdir(results_folder):
    if f.endswith('.csv') and not f.startswith("all_portfolio_histories") and not f.endswith("_ath.csv"):
      lenstr=f.split("_")[-2].split("-")[-1]
      if int(lenstr)>100:
        rdf =  pd.read_csv(os.path.join(results_folder, f))
        if "main_list" in rdf.columns:
          print("--"*10,f,"--"*10)
          ndf = analys_one_df_of_tokens(rdf)
          sorted_combined = analyse_depth_dict(ndf)
          sorted_combined["name"]=f
          print(f'for ${f} sorted_combined is ${sorted_combined}')


          # print(result)
          # print(analyse_counter_list(result["counter_list"]))
          # print(analyse_main_list(result["main_list"]))
          # # e/100
          data_analys_list.append(sorted_combined)

# Create comprehensive results DataFrame
data_analys_df = pd.DataFrame(data_analys_list)

# Display the results
print("Analysis Summary:")
print("=" * 80)
print(data_analys_df.columns)
# print(data_analys_df.to_string(index=False))
l10_10 = ["name"]+list(range(-10, 0))+list(range(1,11))
data_analys_df = data_analys_df[l10_10 ]





In [ ]:
data_analys_df

In [ ]:
for i in merged_df.columns:
  print(i)


In [ ]:
list_of_rows=[]
for i,row in data_analys_df[l10_10].iterrows():
    d = {}
    row = row.dropna()
    for k in row.keys():
      d[k]=int(row[k])
    list_of_rows.append(d)
list_of_rows

In [ ]:
merged_df

In [ ]:
col = ["filename","r_dict","row_res","sizes","strategy_tp","strategy_sl","strategy_buy_again","strategy_max_buy_count","strategy_end_mcap","strategy_dead_coin_market_cap","strategy_migration_market_cap","strategy_buy_again_avg","strategy_sell_tp_on_avg","strategy_sell_sl_on_avg" ]
new_df_row_list = []
def get_dic_from_row(row):
    d = {}
    row = row.dropna()

    for k, v in row.items():
        # Only process if key is numeric (can be negative or positive)
        try:
            k_num = int(k)
        except (ValueError, TypeError):
            continue  # skip non-numeric keys like 'name' or 'sizes'

        # Convert value safely
        if isinstance(v, (int, float)):
            d[k_num] = v
        elif isinstance(v, str) and v.replace('.', '', 1).replace('-', '', 1).isdigit():
            d[k_num] = float(v) if '.' in v else int(v)
        elif isinstance(v, (list, tuple)) and len(v) > 0 and isinstance(v[0], (int, float)):
            d[k_num] = float(v[0])  # or take mean(v) if needed
        # else skip non-numeric/unsupported types

    return d


merged_df["siz_data_analys"]= 0
merged_df["sizes"]= np.nan
merged_df["sizes"]=merged_df["sizes"].astype('object')
merged_df["r_dict"]= pd.NA
merged_df["r_dict"]=merged_df["r_dict"].astype('object')
merged_df["row_res"]= pd.NA
merged_df["row_res"]=merged_df["row_res"].astype('object')
tmdf = merged_df[col]

sizes1 = [1, 1, 1, 1,1, 1, 1, 1,1, 1, 1, 1]  # size_1, size_2, ...
sizes15 = [1, 1.5, 2.25, 3.375,5.06,7.59,11.39,17.08,25.62,38.44,57.66,86.49]  # size_1, size_2, ...
sizes2 = [1, 2, 4, 8,16,32,64,128,256,1024]  # size_1, size_2, ...

sizes18 = [1, 1.8, 3.24, 5.8,10.5,18.9,34 ,61.2,110,198]  # size_1, size_2, ...

data_analys_df = data_analys_df.fillna(0)
for i,row in data_analys_df.iterrows():
    for sizes in [sizes1, sizes15,sizes2,sizes18]:
      mask = merged_df["filename"] == row["name"]
      idx = merged_df.loc[mask].index
      print(idx,idx[0])
      # print(row["name"], idx, len(merged_df[mask]))
      # print(merged_df.loc[mask, col])
      # merged_df.at[idx[0], "sizes"] = sizes # ✅ fixed

      # print(merged_df.loc[merged_df["filename"]==row['name']][col])
      d = get_dic_from_row(row)
      sum_of_values = 0
      number_of_trades = 0

      for k,v in d.items():
        sum_of_values += abs(k*v)
        number_of_trades += v

      ba_factor =merged_df.loc[mask]["strategy_buy_again"].values[0]
      tp_factor =merged_df.loc[mask]["strategy_tp"].values[0]
      sl_factor =merged_df.loc[mask]["strategy_sl"].values[0]

      r = get_row_res( d, sizes=sizes ,  ba_factor= ba_factor, tp_factor=1.2, sl_factor=0.7 )
      pnl = r / sum_of_values

      # merged_df.at[idx[0], "row_res" ] = r
      # merged_df.loc[mask, "siz_data_analys" ] = 1
      print("Merging")
      print(idx)
      print(row.to_dict())
      print(tmdf.loc[mask].to_dict(orient="records"))
      new_df_row_list.append( tmdf.loc[mask].to_dict(orient="records")[0] | { "r_dict":d,"row_res":r ,"pn":pnl,"sum_of_values":sum_of_values,"number_of_trades":number_of_trades, "sizes":sizes[:3]}  | row.to_dict())


In [ ]:
ddf= pd.DataFrame(new_df_row_list)
# ddf.sort_values("row_res")

ddf.sort_values("strategy_max_buy_count")
# ddf.sort_values("pn")



In [ ]:
import pandas as pd

import pandas as pd
import numpy as np

def group_and_average(df):
    """
    Groups rows with same (strategy_max_buy_count, sizes)
    and averages numeric columns (-10..-1, 1..10),
    handling NaN safely and list-type 'sizes'.
    """

    # Convert 'sizes' lists to tuples so they can be grouped
    df["sizes_tuple"] = df["sizes"].apply(lambda x: tuple(x) if isinstance(x, (list, np.ndarray)) else x)

    # Select numeric result columns (e.g., -10 .. -1 and 1 .. 10)
    numeric_cols = [col for col in df.columns if isinstance(col, (int, float))]
    print(numeric_cols)
    # Group by strategy_max_buy_count and sizes_tuple, compute mean ignoring NaN
    grouped = (
        df.groupby(["strategy_max_buy_count", "sizes_tuple"], as_index=False)[numeric_cols]
          .mean(numeric_only=True)
    )

    # Convert tuple back to list (optional, for readability)
    grouped["sizes"] = grouped["sizes_tuple"].apply(list)
    grouped.drop(columns=["sizes_tuple"], inplace=True)

    return grouped

gdf = group_and_average(ddf)
gdf

In [ ]:
new_gdf_row_list=[]
for i,row in gdf.iterrows():
    for sizes in [sizes1, sizes15,sizes2,sizes18]:
      d = get_dic_from_row(row)
      sum_of_values = 0
      number_of_trades = 0

      for k,v in d.items():
        sum_of_values += abs(k*v)
        number_of_trades += v

      ba_factor =0.8
      tp_factor =1.2
      sl_factor =0.7

      r = get_row_res( d, sizes=sizes ,  ba_factor= ba_factor, tp_factor=1.2, sl_factor=0.7 )
      pnl = r / sum_of_values

      # merged_df.at[idx[0], "row_res" ] = r
      # merged_df.loc[mask, "siz_data_analys" ] = 1
      new_gdf_row_list.append(  { "r_dict":d,"row_res":r ,"pn":pnl,"sum_of_values":sum_of_values,  "sizes":sizes[:3]}  | row.to_dict())
ngdf = pd.DataFrame(new_gdf_row_list)
ngdf

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

def draw_dot_chart(df,x_columns = "strategy_migration_market_cap",results_column="row_res"):
    """
    Scatter chart:
      x-axis  -> df["x_columns"] (125k to 1M)
      y-axis  -> 2nd value of df["sizes"] (e.g. 1.8)
      dot size -> proportional to df["results_column"]
      dot color -> df[results_column]
    """

    # Extract 2nd value from each "sizes" list
    df["size_y"] = df["sizes"].apply(
        lambda s: s[1] if isinstance(s, (list, tuple)) and len(s) > 1 else None
    )

    # Handle missing values safely
    df = df.dropna(subset=[x_columns, "size_y", results_column])

    # Normalize "results_column" for dot size scaling
    size_scale = 50
    row_min, row_max = df[results_column].min(), df[results_column].max()
    if row_max == row_min:
        row_max += 1  # avoid division by zero
    dot_sizes = ((df[results_column] - row_min + 1) / (row_max - row_min + 1)) * size_scale * 10

    # Plot
    plt.figure(figsize=(9, 6))
    scatter = plt.scatter(
        df[x_columns],
        df["size_y"],
        s=dot_sizes,
        c=df[results_column],
        cmap="coolwarm",
        alpha=0.75,
        edgecolors="k"
    )

    # plt.xscale("log")  # good for 125k–1M
    plt.xlabel("Market Cap (x_columns)")
    plt.ylabel("Second value of sizes (size_y)")
    plt.title("Market Cap vs. Sizes Ratio (2nd value) — Dot size & color = results_column")
    plt.colorbar(scatter, label="results_column")
    plt.grid(True, linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.show()
draw_dot_chart(ngdf,results_column="pn",x_columns="strategy_max_buy_count")
draw_dot_chart(ngdf,results_column="row_res",x_columns="strategy_max_buy_count")

In [ ]:
ddf[ddf["filename"].str.startswith(("Base"))].sort_values("row_res")


In [ ]:
ddf[ddf["filename"].str.startswith(("After"))].sort_values("row_res")


In [ ]:

import ast



rdf = pd.read_csv(results_folder+"BaseBuySell20_30_MartingaleSizer_2025-10-20_15:46:38_len-306_memes.csv")

ndf = analys_one_df_of_tokens(rdf)
sorted_combined = analyse_depth_dict(ndf)
# print(f'for ${file} sorted_combined is ${sorted_combined}')

def sum_minus(d):
    s = 0
    for k,v in d.items():
      if k < 0:
        s = s+ k*v
    return s
sum_minus(sorted_combined)

In [ ]:
ndf["depth_dict"]


from collections import Counter

combined = Counter()

for d in ndf["depth_dict"]:
    combined.update(d)

sorted_combined = dict(sorted(combined.items()))
print(sorted_combined)

def sum_minus(d):
    s = 0
    for k,v in d.items():
      if k < 0:
        s = s+ k*v
    return s
sum_minus(sorted_combined)


In [ ]:
import matplotlib.pyplot as plt

# Assuming sorted_combined from previous step
keys = list(sorted_combined.keys())
counts = list(sorted_combined.values())

plt.figure(figsize=(10, 5))
plt.bar(keys, counts, color='skyblue')
plt.xlabel("Trade Depth (positive=TP, negative=SL)")
plt.ylabel("Count")
plt.title("Combined Depth Counts Across All Coins")
plt.xticks(keys)  # show all keys on x-axis
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


In [ ]:
def find_row(summary_df,name):
  print("looking for",name+"_memes.csv")
  row = summary_df[summary_df["filename"]==name+"_memes.csv"]
  print(row)
# summary_df
#  [filename, total_trades, profitable_trades, profitable_percentage, total_start_value, total_final_value, total_pnl, pnl_percentage, none_count, none_percentage, sl_count, sl_percentage, tp_count, tp_percentage, mean_final_value, max_final_value, min_final_value, std_final_value, median_final_value]
# details_df[['strategy', 'sizer', 'time', 'len',
#        'strategy_tp', 'strategy_sl', 'strategy_buy_again',
#        'strategy_max_buy_count', 'strategy_end_mcap', 'strategy_rsi',
#        'strategy_dead_coin_market_cap', 'strategy_migration_market_cap',
#        'strategy_buy_again_avg', 'strategy_sell_tp_on_avg',
#        'strategy_sell_sl_on_avg', 'sizer_stake_cash', 'sizer_multiplier',
#        'sizer_max_multiplier', 'sizer_percentage', 'source_file']]
find_row(summary_df,details_df["source_file"].values[0])

In [ ]:
merged_df

In [ ]:
# merged_df = merged_df[merged_df["total_trades"]>5]
# merged_df["total_final_value"] = merged_df["total_final_value"]/30600
# merged_df["total_pnl"] = merged_df["total_pnl"]/30600

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

merged_df = merged_df[['filename',   'profitable_trades',
       'profitable_percentage', 'total_final_value',
       'total_pnl', 'pnl_percentage',
       'sl_count', 'sl_percentage', 'tp_count', 'tp_percentage',
       'mean_final_value', 'max_final_value', 'min_final_value',
       'std_final_value', 'median_final_value',
         'strategy_tp', 'strategy_sl', 'strategy_buy_again',
       'strategy_max_buy_count', 'strategy_rsi',
        'strategy_buy_again_avg', 'strategy_sell_tp_on_avg',
       'strategy_sell_sl_on_avg',  'sizer_multiplier' ]]
merged_df

In [ ]:
merged_df

In [ ]:
merged_df[(merged_df["strategy_buy_again_avg"]==0)&(merged_df[ "sizer_multiplier"]==1.5)].sort_values("mean_final_value")[["pnl_percentage","mean_final_value","median_final_value","strategy_tp","strategy_sl","strategy_buy_again","strategy_max_buy_count","sizer_multiplier"]]
merged_df[(merged_df["strategy_buy_again_avg"]==0)&(merged_df[ "sizer_multiplier"]==1.5)].sort_values("strategy_tp")[["pnl_percentage","mean_final_value","median_final_value","strategy_tp","strategy_sl","strategy_buy_again","strategy_max_buy_count","sizer_multiplier"]]

In [ ]:
  1.20	0.6	0.8	4	1.5
  1.20	0.6	0.9	6	1.5

 	1.25	0.6	0.8	6	1.5
 	1.25	0.6	0.8	4	1.5
 	1.25	0.6	0.7	4	1.5


In [ ]:
print(csv_files_1s[-1])
df =ready_df(pd.read_csv(csv_files_1s[-1]))
ath_index = df["close"].idxmax()
ath = df["close"].max()
df_to_run = df.loc[ath_index + 1:]
df
# (df["close"].iloc[-1] * 1_000_000_000- df["close"].iloc[0]* 1_000_000_000)